# Milestone 5 — LLMs & Fine-tuning

1. Review summarizer → structured pros/cons
2. Grounded QA assistant with retrieval over reviews

In [ ]:
import random
import pandas as pd
from transformers import pipeline

from src.data import load_splits
from src.inference import answer_product_question, summarize_product_reviews
from src.utils import setup_notebook_path

setup_notebook_path()

In [ ]:
splits = load_splits()
reviews = splits["train_reviews"]
products = splits["train_products"]

sample_pid = reviews["product_id"].value_counts().index[0]
sample_title = products.loc[products["product_id"] == sample_pid, "title"].iloc[0]
print("Product:", sample_title)

zero_shot = summarize_product_reviews(sample_pid, mode="zero_shot")
grounded = summarize_product_reviews(sample_pid, mode="grounded")
print("\n--- Zero-shot style ---\n", zero_shot)
print("\n--- Grounded ---\n", grounded)

In [ ]:
question = "Does this product work for sensitive skin?"
answer = answer_product_question(sample_pid, question)
print(question)
print(answer)

In [ ]:
# Optional: lightweight fine-tune demo on pseudo summary pairs (skip if low on time)
pairs = []
for pid in reviews["product_id"].value_counts().head(50).index:
    subset = reviews[reviews["product_id"] == pid].head(10)
    source = " ".join(subset["review_text"].tolist())[:1500]
    pos = subset[subset["rating"] >= 4]["review_text"].head(1).tolist()
    neg = subset[subset["rating"] <= 2]["review_text"].head(1).tolist()
    target = f"Pros: {pos[0][:120] if pos else 'Good quality.'} Cons: {neg[0][:120] if neg else 'None noted.'}"
    pairs.append({"source": source, "target": target})

print(f"Built {len(pairs)} fine-tune pairs (demo). Compare zero-shot vs fine-tuned in report.")